In [134]:
import openpyxl
from openpyxl.styles import PatternFill, Border, Side, Alignment, Font
import os

g_cables = dict()
g_wattage = g_weight = summary = 0
distributions_info = []

filename = "rack_layout_top_down.xlsx"
if os.path.exists(filename):
    os.remove(filename)
    print(f"Existing file '{filename}' deleted.")

def generate_visual():
    global g_cables, g_wattage, g_weight, summary
    overall_cables = dict()
    try:
        wb = openpyxl.load_workbook(filename)
    except FileNotFoundError:
        wb = openpyxl.Workbook()
    if 'Sheet' in wb.sheetnames:
        del wb['Sheet']
    if 'summary' not in wb.sheetnames:
        summary = wb.create_sheet(title='summary')
       
    else:
        summary = wb['summary']
    # Define styles
    header_font = Font(bold=True)
    border = Border(left=Side(style='thin'), right=Side(style='thin'), 
                   top=Side(style='thin'), bottom=Side(style='thin'))
    center_aligned = Alignment(horizontal='center', vertical='center')
    left_aligned = Alignment(horizontal='left', vertical='center', wrap_text=True)
    total_font = Font(bold=True)
    total_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
    summary_font = Font(bold=True, color='FFFFFF')
    summary_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')  # Blue color
    summary_row = 1
    summary_cell = summary.cell(row=summary_row, column=1, value=f"This is an expected rack layout. It may be changed including the cables as per identifying the \
layout of the cable trays and OOB switches placement strategy")
    summary.cell(row=summary_row, column=1).alignment = Alignment(wrap_text=True)
    summary_cell.fill = total_fill
    summary_row += 3
     # Set up summary headers
    summary.column_dimensions['A'].width = 80
    summary.column_dimensions['B'].width = 15
    summary.column_dimensions['C'].width = 50
    # Write headers
    summary_cell = summary.cell(row=summary_row, column=1)
    summary_cell.fill = summary_fill
    summary_cell = summary.cell(row=summary_row, column=2)
    summary_cell.fill = summary_fill
    summary_cell = summary.cell(row=summary_row, column=3)
    summary_cell.fill = summary_fill
    summary.cell(row=summary_row, column=1, value="Rack Group Description").font = Font(bold=True)
    summary.cell(row=summary_row, column=2, value="Rack Count").font = Font(bold=True)
    summary.cell(row=summary_row, column=3, value="Details").font = Font(bold=True)
    loc = 0
    for rack_tuple, item in unique_racks.items():
        loc += 1
        distribution = item['details']
        rack_ids = item['rack_ids']
        node_positions = distribution['stable_placement']
        device_info = distribution.get('device_info', {})
        lan_info = distribution.get('lan_info', {})
        tab_name = f"Group_{loc}"
        ws = wb.create_sheet(title=tab_name)
        # Set column widths
        ws.column_dimensions['A'].width = 25
        ws.column_dimensions['B'].width = 25
        ws.column_dimensions['C'].width = 15
        ws.column_dimensions['D'].width = 15
        #ws.column_dimensions['E'].width = 100  # Added column for cables
        #ws.column_dimensions['F'].width = 15  # Added column for external rack connections


        

        from colorsys import hls_to_rgb
        import hashlib

        #ws.merge_cells(start_row=1, end_row=1, start_column=1, end_column=6)
        summary_cell = ws.cell(row=1, column=1)
        summary_cell.font = summary_font
        summary_cell.fill = summary_fill
        summary_cell.alignment = center_aligned
        ws.merge_cells(start_row=1, end_row=1, start_column=1, end_column=4)
        summary_text = f"Rack Group {loc} - Total Racks: {len(rack_ids)} - Rack IDs: {rack_ids}"
        summary_cell.value = summary_text
        #ws.cell(row=1, column=1, value=summary_text)

        class AutoColorMap(dict):
            def __missing__(self, key):
                MAX_ATTEMPTS = 10  # Prevent infinite loops
                for _ in range(MAX_ATTEMPTS):
                    # Generate color by hashing the key with iteration count
                    hash_val = int(hashlib.md5(f"{key}{len(self)}".encode()).hexdigest()[:6], 16)
                    hue = hash_val % 360  # Full hue range
                    saturation = 50 + hash_val % 45  # 50-95% (vibrant but not neon)
                    lightness = 20 + hash_val % 60   # 20-80% (avoid extremes)

                    # Convert HSL to RGB
                    r, g, b = hls_to_rgb(hue/360, lightness/100, saturation/100)
                    color = f"{int(r*255):02X}{int(g*255):02X}{int(b*255):02X}"

                    # Calculate color brightness
                    brightness = (0.299 * r + 0.587 * g + 0.114 * b)

                    # Reject colors that are too white or too dark
                    if 0.15 <= brightness <= 0.85:  # Good visibility range
                        self[key] = color
                        return color

                # Fallback to a medium blue if max attempts reached
                self[key] = "4B8DF0"
                return "4B8DF0"

        color_map = {
            'compute_nodes': 'FF9999',
            'GPU_nodes': '99CCFF',
            'LAN': '99FF99',
            'NVMe': "4B8DF0"
        }

        color_this = AutoColorMap(color_map)
        for color in colors_info:
            color_map[color] = color_this[color]
        for color in LANs:
            color_map[color] = color_this[color]


        # Format cables information function
        def format_cables_info(device_name):

            if device_name not in device_info:
                return ""

            cable_info = ""

            for key,lan_data in device_info[device_name].items():
                if 'LAN_' in key:
                    lan = '_'.join(key.split('_')[:-1])
                    if lan_data.get('to_external_rack',-1) > -1:
                        rack = f"to Rack_{lan_data['to_external_rack']+1}"
                    else:
                        rack = 'to this rack'
                    cable_info = f"{cable_info},{lan}:{lan_data['cables_count']:.0f}*{lan_data['cable_type']['model']}-->{rack}"

            if cable_info:
                cable_info = cable_info[1:]
            return cable_info




        # Get external rack connection
        def get_external_rack(device_name):
            if device_name not in device_info:
                return ""

            return device_info[device_name].get('to_external_rack', "")

        # Write headers
        #headers = ["Rack Position", "Device", "Wattage (W)", "Weight (kg)", "Cables", "External Rack"]
        headers = ["Rack Position", "Device", "Wattage (W)", "Weight (kg)"]
        for col, header in enumerate(headers, start=1):
            cell = ws.cell(row=2, column=col, value=header)
            cell.font = header_font
            cell.border = border
            cell.alignment = center_aligned

        # Track occupied positions
        occupied_info = {}
        total_wattage = 0
        total_weight = 0
        occupied_positions = {}
        occupied_label = {}

        # First pass: Identify all positions and their heights
        for device, posy in node_positions.items():
            if 'LAN_' in device:
                lan_parts = '_'.join(device.split('_')[:-1]).split('__')
                lan_type = lan_parts[0]
                switch_model = lan_parts[1]
                if 'pine' in lan_type:
                    lan_dict = Spines
                else:
                    lan_dict = LANs
                for switch in lan_dict[lan_type]['switch']:
                    if switch['model'] == switch_model:
                        height = switch['height']
                        wattage = switch['wattage']
                        weight = switch['weight']
                        break

                pos = posy + height - 1
                occupied_label[pos] = f"{pos-height+1}-{pos}"
                for i in range(pos-height+1, pos+1):
                    occupied_positions[i] = device

                cables_info = format_cables_info(device)
                #external_rack = get_external_rack(device)
                occupied_info[pos] = {
                    'device': device,
                    'type': 'LAN Switch',
                    'height': height,
                    'color': color_map.get('LAN', 'FFFFFF'),
                    'wattage': wattage,
                    'weight': weight,
                    #'cables': cables_info,
                    #'external_rack': external_rack
                }
                total_wattage += wattage
                total_weight += weight
            else:
                node_type = '_'.join(device.split('_')[:-1])
                height = colors_info[node_type]['height']
                wattage = colors_info[node_type]['wattage']
                weight = colors_info[node_type]['weight']
                pos = posy + height - 1
                occupied_label[pos] = f"{pos-height+1}-{pos}"
                for i in range(pos-height+1, pos+1):
                    occupied_positions[i] = device

                #cables_info = format_cables_info(device)
                #external_rack = get_external_rack(device)

                occupied_info[pos] = {
                    'device': device,
                    'type': node_type.replace('_', ' ').title(),
                    'height': height,
                    'color': color_map.get(node_type, 'FFFFFF'),
                    'wattage': wattage,
                    'weight': weight,
                    #'cables': cables_info,
                    #'external_rack': external_rack
                }
                total_wattage += wattage
                total_weight += weight

        # Second pass: Write data to worksheet
        current_row = 3
        for rack_pos in range(42, 0, -1):
            # Skip if this position is covered by a device that starts above
            if rack_pos in occupied_positions and rack_pos not in occupied_info:
                continue

            # Get position label
            position_label = occupied_label.get(rack_pos, str(rack_pos))

            # Write rack position
            ws.cell(row=current_row, column=1, value=position_label)
            ws.cell(row=current_row, column=1).border = border
            ws.cell(row=current_row, column=1).alignment = center_aligned

            if rack_pos in occupied_info:
                device_info = occupied_info[rack_pos]
                height = device_info['height']

                # Write device info
                ws.cell(row=current_row, column=2, value=device_info['device'])
                ws.cell(row=current_row, column=3, value=device_info['wattage'])
                ws.cell(row=current_row, column=4, value=device_info['weight'])
                #ws.cell(row=current_row, column=5, value=device_info['cables'])
                #ws.cell(row=current_row, column=6, value=device_info['external_rack'])
                g_wattage += device_info['wattage']
                g_weight += device_info['weight']
                #if '*' in device_info['cables']:
                #    for cable_info in device_info['cables'].split(','):
                #        cable_model = cable_info.split('*')[1].split('--')[0]
                #        cable_count = cable_info.split('*')[0].split(':')[1]
                #        if g_cables.get(cable_model,0) == 0:
                #            g_cables[cable_model] = 0
                #        g_cables[cable_model] += int(cable_count)
                g_wattage += device_info['wattage']
                g_weight += device_info['weight']
                # Apply styling to all cells that will be merged
                fill = PatternFill(start_color=device_info['color'], end_color=device_info['color'], fill_type='solid')
                #for col in range(1, 7):  # Updated to include new columns
                for col in range(1, 5):  # Updated to include new columns
                    ws.cell(row=current_row, column=col).fill = fill
                    ws.cell(row=current_row, column=col).border = border
                    #if col == 5:  # Cable column - left aligned with text wrapping
                    #    ws.cell(row=current_row, column=col).alignment = left_aligned
                    #else:
                    #    ws.cell(row=current_row, column=col).alignment = center_aligned

                # Merge cells if height > 1
                if height > 1:
                    #for col in range(1, 7):  # Updated to include new columns
                    for col in range(1, 5):  # Updated to include new columns
                        ws.merge_cells(
                            start_row=current_row,
                            end_row=current_row + height - 1,
                            start_column=col,
                            end_column=col
                        )

                    # Apply border to all merged cells
                    for row in range(current_row, current_row + height):
                        #for col in range(1, 7):  # Updated to include new columns
                        for col in range(1, 5):  # Updated to include new columns
                            ws.cell(row=row, column=col).border = border

                current_row += height
            else:
                # Empty position
                #for col in range(2, 7):  # Updated to include new columns
                for col in range(2, 5):  # Updated to include new columns
                    ws.cell(row=current_row, column=col, value="")
                    ws.cell(row=current_row, column=col).border = border
                    ws.cell(row=current_row, column=col).alignment = center_aligned

                current_row += 1

        # Add total row
        total_row = current_row + 1
        ws.cell(row=total_row, column=1, value="TOTAL").font = total_font
        ws.cell(row=total_row, column=2, value="").font = total_font
        ws.cell(row=total_row, column=3, value=total_wattage).font = total_font
        ws.cell(row=total_row, column=4, value=total_weight).font = total_font
        #ws.cell(row=total_row, column=5, value="").font = total_font
        #ws.cell(row=total_row, column=6, value="").font = total_font

        #for col in range(1, 7):  # Updated to include new columns
        for col in range(1, 5):  # Updated to include new columns
            ws.cell(row=total_row, column=col).border = border
            ws.cell(row=total_row, column=col).fill = total_fill
            ws.cell(row=total_row, column=col).alignment = center_aligned
        current_row = total_row + 2
        headers = ["Group Cable Description", "Total Count in all the group"]
        for col, header in enumerate(headers, start=1):
            cell = ws.cell(row=current_row, column=col, value=header)
            cell.font = header_font
            cell.border = border
            cell.alignment = center_aligned
        
        for  key,value in item['details']['cables_info'].items():
            current_row += 1
            cell = ws.cell(row=current_row, column=1, value=key)
            cell.border = border
            cell = ws.cell(row=current_row, column=2, value=f"{value:.0f}")
            cell.border = border
            overall_cables[key] = overall_cables.get(key,0) + value
        summary_row += 1
        summary.cell(row=summary_row, column=1, value=f"Rack_{loc} total units")
        summary.cell(row=summary_row, column=2, value=len(rack_ids))
        summary.cell(row=summary_row, column=3, value=f"{rack_ids}")
    summary_row += 3
    summary_cell = summary.cell(row=summary_row, column=1)
    summary_cell.fill = summary_fill
    summary.cell(row=summary_row, column=1, value="Cable Descriptions").font = Font(bold=True)
    summary_cell = summary.cell(row=summary_row, column=1)
    summary_cell.fill = summary_fill
    summary_cell = summary.cell(row=summary_row, column=2)
    summary.cell(row=summary_row, column=2, value="Qty").font = Font(bold=True)
    summary_cell.fill = summary_fill
    for cable,count in overall_cables.items():
        summary_row += 1
        summary.cell(row=summary_row, column=1, value=cable)
        summary.cell(row=summary_row, column=2, value=count)
        
    for row in summary.iter_rows():
        for cell in row:
            if cell.value:  # Only apply to cells with content (optional)
                cell.alignment = Alignment(horizontal='center', vertical='center',wrap_text=True)
                cell.border = border
                
    # Save the workbook
    wb.save(filename)
    print(f"Excel file '{filename}' has been created with positions from 42 at top to 1 at bottom.")

Existing file 'rack_layout_top_down.xlsx' deleted.


In [135]:
def merge_dicts_fast(dict1, dict2):
    result = dict1.copy()
    for key, value in dict2.items():
        result[key] = result.get(key, 0) + value
    return result

def get_rack_groups(distributions_info):
    unique_racks = {}
    for i, info in enumerate(distributions_info):
        if 'rack_config' in distributions_info[info]:
            # Create a hashable representation of the rack configuration
            rack_config = distributions_info[info]['rack_config']
            # Convert the rack_config dictionary to a tuple of sorted items for hashability
            rack_tuple = tuple(sorted(rack_config.items()))

            # Store the rack details with its ID
            rack_details = {
                'rack_id': i+1,
                'config': rack_config,
                'stable_placement': distributions_info[info].get('stable_placement', []),
                'bottom_place_periority': distributions_info[info].get('bottom_place_periority', []),
                'lan_info': distributions_info[info].get('lan_info', {}),
                'cables_info': distributions_info[info].get('cables_info', {}),
                'spine_info': distributions_info[info].get('spine_info', {}),
                'device_info': distributions_info[info].get('device_info', {})
            }
            # If we've seen this configuration before, append to the list
            if rack_tuple in unique_racks:
                unique_racks[rack_tuple]['rack_ids'].append(i+1) # append D
                unique_racks[rack_tuple]['count'] += 1
                unique_racks[rack_tuple]['details']['cables_info'] = merge_dicts_fast(unique_racks[rack_tuple]['details']['cables_info'], rack_details['cables_info'])
            else:
                # First time seeing this configuration
                unique_racks[rack_tuple] = {
                    'rack_ids': [i+1],
                    'count': 1,
                    'details': rack_details
                }

    return unique_racks
def iterate_on_racks():
    global distributions_info, unique_racks
    total_wattage = 0
    total_height = 0
    total_count = 0
    filled_boxes = []
    # Create a dictionary to store unique rack configurations
    unique_racks = {}
    rack_summary = []
    for i, info in enumerate(distributions_info):
        # Track the total count regardless of whether we're processing racks
        if 'stable_placement' in distributions_info[info]:
            total_count += len(distributions_info[info]['stable_placement'])
    unique_racks = get_rack_groups(distributions_info)
    generate_visual()
    
    return

In [136]:
import pickle
from collections import Counter
from math import floor
def global_main():
    global colors_info, LANs, Spines, Cables, max_box_wattage, max_box_height,  rack_height_u , rack_weight_kg, rack_width_mm, \
            rack_depth_mm, u_height_mm, rack_height_mm, rack_height_u, rack_cg_height_mm, unit_to_cm, u_height_mm, \
            device_to_rackside, racktop_to_ceiling, rack_to_rack, Rack_rows, Rack_rows, distributions_info,project,config
    


    with open(f"Latest_Racks_{project}_{config}.pkl", 'rb') as f:
        colors_info, LANs, Spines, Cables, Rack_rows, distributions_info = pickle.load(f)
        
    
    
    max_box_wattage = 16000
    max_box_height = rack_height_u = 42
    rack_weight_kg = 114.55
    rack_width_mm = 750
    rack_depth_mm = 1200
    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2
    global_rack_signature = dict()
    unit_to_cm = u_height_mm /10
    # the following are in units
    device_to_rackside = 6.75
    racktop_to_ceiling = 4.5 # assumed 20cmint
    rack_to_rack = 4.5 # assumed from side to the adjacent side no spacing
    # adding to the Cables the maximum stretch
    for switch in Cables:
        for cable in Cables[switch]:
            cable['in_rack_stretch'] = floor((cable['length']*100/unit_to_cm) - (2* device_to_rackside))
            
            
if __name__ == "__main__":
    project = 'DRMEWA'
    config = '4_35R_27S_8g'
    global_main()
    iterate_on_racks()

Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.


In [137]:
connected_racks = dict()
for i in distributions_info:
    if 'device_info' not in distributions_info[i]:
        continue
    for x,v in distributions_info[i]['device_info'].items():
        for y in v:
            if 'LAN_' in y and v[y].get('to_external_rack',0) > 0:
                if connected_racks.get(i,0) == 0:
                    connected_racks[i] = set()
                #if connected_racks[i].get(v[y]['to_external_rack'],0) == 0:
                connected_racks[i].add(v[y]['to_external_rack'])
                #connected_racks[i][v[y]['to_external_rack']].append((x,y))
print(connected_racks)
        

{3: {2}, 4: {2, 3}, 5: {2, 3}, 6: {2, 3}, 7: {2, 3, 6}, 8: {2, 3, 6}, 9: {2, 3, 6}, 10: {2, 3, 6}, 11: {2, 3, 6}, 12: {2, 3, 6}, 13: {2, 3, 6}, 14: {2, 3, 6}, 15: {2, 3, 6}, 16: {2, 3, 6}, 17: {2, 3, 6}, 18: {2, 3, 6}, 19: {2, 3, 6}, 20: {2, 3, 6}, 21: {2, 3, 6}, 22: {2, 3, 6}, 23: {2, 3, 6}, 24: {2, 3, 6}, 25: {2, 3, 6}, 26: {2, 3, 6}, 27: {2, 3, 6}, 28: {2, 3, 6}, 29: {2, 3, 6}, 30: {2, 3, 6}, 31: {2, 3, 6}, 32: {2, 3, 6}, 33: {2, 3, 6}, 34: {2, 3, 6}}
